# 🤖 Agentic AI Professional Course
## Module 2 — Prompt Fundamentals: Hands-On Exercises

---

> **Welcome.** This notebook contains all four lab exercises for Module 2. Each exercise is self-contained and corresponds to one hour of the live session. Complete them in order — each builds on the previous.

### What you'll build in this notebook
| Hour | Exercise | What you build |
|------|----------|----------------|
| Hour 4 | Prompt Anatomy Workshop | Diagnose and rewrite 3 broken prompts using the 6-element framework |
| Hour 5 | Prompt Type Classifier | Classify 10 prompts by type and validate using an LLM judge |
| Hour 6 | Zero-Shot vs Few-Shot Lab | Compare techniques on a real classification task + build a decomposition pipeline |
| Hour 7 | Iterative Refinement Loop | Build a full critique-and-rewrite Python loop with stopping conditions |

### Prerequisites
- An Anthropic API key (get one at https://console.anthropic.com)
- Basic Python knowledge (you already have this)

---

**💡 How to use this notebook:**
- Read each markdown cell before running code
- Fill in sections marked with `# YOUR CODE HERE` or `# YOUR ANSWER HERE`
- Run the validation cells to check your work
- The `🔍 DEBRIEF` sections reveal the expected answers — try first!

## ⚙️ Setup — Run This First

Install the Anthropic SDK and configure your API key.

In [ ]:
# Install the Anthropic SDK
!pip install anthropic --quiet
print("✅ anthropic installed")

In [ ]:
import anthropic
import json
import time
from textwrap import dedent

# ─────────────────────────────────────────────────
# ADD YOUR API KEY HERE
# Get one at: https://console.anthropic.com
# ─────────────────────────────────────────────────
ANTHROPIC_API_KEY = "sk-ant-..."  # ← Replace this

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-haiku-4-5-20251001"  # Fast + cost-effective for exercises

def call_llm(prompt: str, system: str = None, max_tokens: int = 1024) -> str:
    """Simple wrapper around the Anthropic API."""
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text

# Quick connectivity test
test = call_llm("Reply with exactly: READY")
print(f"API connection: {test.strip()}")
print(f"Model: {MODEL}")

---
# 🔵 Hour 4 Exercise — Prompt Anatomy Workshop

## The Six-Element Framework

Every well-engineered prompt contains some combination of these six elements:

```
① ROLE        — Who the model is pretending to be
② TASK        — What it must do (starts with an action verb)
③ CONTEXT     — Background it needs (can't assume)
④ CONSTRAINTS — What it must/must not do
⑤ OUTPUT FMT  — The exact shape of the response
⑥ EXAMPLES    — Show, don't just tell (few-shot)
```

In this exercise, you will:
1. Analyse three broken prompts and identify what's missing
2. Rewrite each one using the full framework
3. Compare outputs: broken vs. structured
4. Use an LLM judge to score your rewrites

### 4.1 — Diagnose the Broken Prompts

Below are three prompts that a developer might write instinctively. Before rewriting, diagnose **which elements are missing** from each.

In [ ]:
# The three broken prompts
broken_prompts = {
    "P1": "Tell me about machine learning.",
    "P2": "Summarise this document.",
    "P3": "Write some Python code.",
}

# ─────────────────────────────────────────────────────────────────────────────
# TASK: For each prompt, list which of the 6 elements are MISSING.
# Use: role, task, context, constraints, output_format, examples
# ─────────────────────────────────────────────────────────────────────────────

your_diagnosis = {
    "P1": [
        # YOUR ANSWER HERE — list missing elements as strings
        # Example: "role", "context", ...
    ],
    "P2": [
        # YOUR ANSWER HERE
    ],
    "P3": [
        # YOUR ANSWER HERE
    ],
}

print("Your diagnosis:")
for key, missing in your_diagnosis.items():
    print(f"  {key}: {broken_prompts[key]}")
    print(f"       Missing: {missing or '(not filled in yet)'}")
    print()

In [ ]:
# ─── REVEAL: Expected diagnosis ──────────────────────────────────────────────
# Run this cell AFTER completing your own diagnosis above.

expected_diagnosis = {
    "P1": ["role", "context", "constraints", "output_format"],
    "P2": ["role", "context", "constraints", "output_format"],  # also missing the actual document!
    "P3": ["role", "task", "context", "constraints", "output_format"],  # 'write' is vague, not specific
}

print("🔍 DEBRIEF — Expected diagnosis:")
print("─" * 55)
for key, missing in expected_diagnosis.items():
    print(f"\n{key}: '{broken_prompts[key]}'")
    print(f"  Missing: {missing}")
    
print("\n" + "─" * 55)
print("\nNote: P2 is especially broken — it can't be executed at all")
print("without the actual document. Context isn't just background —")
print("it's the INPUT DATA the model needs.")

### 4.2 — Rewrite Using the Six-Element Framework

Now rewrite all three prompts. Be specific. Invent context where needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK: Fill in your structured rewrites below.
#
# Template for each:
#   [ROLE] ...
#   [TASK] ...
#   [CONTEXT] ...
#   [CONSTRAINTS] ...
#   [OUTPUT FORMAT] ...
#   [EXAMPLE] ... (optional)
# ─────────────────────────────────────────────────────────────────────────────

your_rewrites = {

    "P1_rewritten": """
    [ROLE] ...
    [TASK] ...
    [CONTEXT] ...
    [CONSTRAINTS] ...
    [OUTPUT FORMAT] ...
    """,  # Replace the ... with your content

    "P2_rewritten": """
    [ROLE] ...
    [TASK] ...
    [CONTEXT] ...
    [CONSTRAINTS] ...
    [OUTPUT FORMAT] ...
    """,

    "P3_rewritten": """
    [ROLE] ...
    [TASK] ...
    [CONTEXT] ...
    [CONSTRAINTS] ...
    [OUTPUT FORMAT] ...
    """,
}

print("Your rewrites:")
for key, prompt in your_rewrites.items():
    print(f"\n{'='*60}")
    print(f"{key}:")
    print(prompt)

In [ ]:
# ─── Run this after filling in your rewrites ─────────────────────────────────
# Compare: broken prompt output vs. your structured prompt output

print("📊 COMPARISON: Broken vs. Structured")
print("=" * 70)

comparisons = [
    ("P1", broken_prompts["P1"], your_rewrites["P1_rewritten"]),
    ("P3", broken_prompts["P3"], your_rewrites["P3_rewritten"]),
]

for label, broken, structured in comparisons:
    if "..." in structured:  # Skip if not filled in
        print(f"\n⚠️  {label}: Fill in your rewrite first!")
        continue

    print(f"\n{'─'*70}")
    print(f"🔴 BROKEN: '{broken}'")
    broken_output = call_llm(broken)
    print(f"Output ({len(broken_output)} chars):")
    print(broken_output[:400] + ("..." if len(broken_output) > 400 else ""))

    print(f"\n🟢 STRUCTURED:")
    structured_output = call_llm(structured)
    print(f"Output ({len(structured_output)} chars):")
    print(structured_output[:400] + ("..." if len(structured_output) > 400 else ""))
    time.sleep(1)

In [ ]:
# ─── Auto-score your rewrites using an LLM judge ─────────────────────────────

JUDGE_SYSTEM = """You are a prompt engineering expert who evaluates prompt quality.
You are strict, direct, and do not give credit for incomplete work."""

JUDGE_PROMPT = """
Evaluate this prompt rewrite against the six-element framework.

Score each present element as PRESENT (1) or MISSING (0):
1. ROLE: Does it specify who the model is?
2. TASK: Does it start with a clear action verb?
3. CONTEXT: Does it provide necessary background?
4. CONSTRAINTS: Does it specify what to/not to do?
5. OUTPUT_FORMAT: Does it specify the shape of the response?
6. SPECIFICITY: Is it specific enough to run unsupervised?

Prompt to evaluate:
{prompt}

Return ONLY this JSON (no other text):
{{
  "role": 0,
  "task": 0,
  "context": 0,
  "constraints": 0,
  "output_format": 0,
  "specificity": 0,
  "total": 0,
  "feedback": "one sentence of specific feedback"
}}
"""

print("🧑‍⚖️  LLM JUDGE — Scoring your rewrites")
print("=" * 55)

for key, prompt in your_rewrites.items():
    if "..." in prompt:
        print(f"\n⚠️  {key}: Not filled in yet.")
        continue

    result = call_llm(
        JUDGE_PROMPT.format(prompt=prompt),
        system=JUDGE_SYSTEM
    )

    try:
        # Strip markdown code fences if present
        clean = result.strip().replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean)
        total = scores.get("total", sum(v for k, v in scores.items() if k not in ["total", "feedback"]))
        print(f"\n{key}:")
        print(f"  Score: {total}/6")
        bar = "█" * total + "░" * (6 - total)
        print(f"  [{bar}]")
        for elem in ["role", "task", "context", "constraints", "output_format", "specificity"]:
            icon = "✅" if scores.get(elem) else "❌"
            print(f"  {icon} {elem}")
        print(f"  💬 {scores.get('feedback', '')}")
    except json.JSONDecodeError:
        print(f"\n{key}: {result}")
    time.sleep(1)

In [ ]:
# ─── REVEAL: Model answer for P1 ─────────────────────────────────────────────

p1_model_answer = """
[ROLE]
You are a senior machine learning engineer with 10 years of industry experience
at tech companies. You explain complex ML concepts clearly to professional
software engineers who know Python well but have no ML background.

[TASK]
Explain the core concepts of supervised machine learning.

[CONTEXT]
The reader is a backend Python developer who regularly builds REST APIs and
works with databases. They have never worked with ML models but are considering
integrating a pre-trained model into their product.

[CONSTRAINTS]
- Use Python analogies where possible (functions, dictionaries, loops)
- Do not use mathematical notation or Greek letters
- Do not cover unsupervised or reinforcement learning
- Maximum 250 words

[OUTPUT FORMAT]
Return three sections:
1. "The Core Idea" — one paragraph, plain language
2. "A Python Analogy" — one code comment block (no runnable code)
3. "What This Means For You" — two bullet points on practical implications
"""

print("🔍 DEBRIEF — Model answer for P1 (Machine Learning)")
print("─" * 60)
print(p1_model_answer)
print("\n" + "─" * 60)
print("Running model answer to show the expected output quality...\n")
output = call_llm(p1_model_answer)
print(output)

---
# 🔵 Hour 5 Exercise — Prompt Type Classifier

## The Six Prompt Types

| Type | Primary Role | Output |
|------|-------------|--------|
| **Instruction** | Execute a task | Task result |
| **Role** | Define model identity | (sets behaviour) |
| **Planning** | Decompose a goal | Subtask list |
| **Tool-Use** | Call external tools | Tool call + result |
| **Routing** | Classify input | Category label |
| **Evaluation** | Judge quality | Score + feedback |

In this exercise, you will:
1. Classify 10 prompts by type (manually, before checking)
2. Use an LLM classifier to auto-classify them
3. Compare your answers to the LLM's and the expected answers
4. Write your own prompt of each type

In [ ]:
# The 10 prompts to classify
prompts_to_classify = [
    (1,  "You are a legal assistant specialising in UK contract review."),
    (2,  "Summarise the key points from this meeting transcript."),
    (3,  "Break down this goal into subtasks: Launch a product blog for our SaaS platform."),
    (4,  "You have access to a calculator tool. Use it to compute the compound interest on £10,000 at 5% over 3 years."),
    (5,  "Determine whether this support ticket is: BILLING, TECHNICAL, or ACCOUNT."),
    (6,  "Score this product description from 1-10 on persuasiveness and clarity. Give specific feedback."),
    (7,  "Extract all named entities from this news article and return them as a JSON list."),
    (8,  "What steps would I need to take to set up a RAG system from scratch?"),
    (9,  "Is this customer complaint urgent? Respond with YES, NO, or UNCLEAR only."),
    (10, "Review this Python function and identify any bugs, anti-patterns, or performance issues."),
]

VALID_TYPES = ["instruction", "role", "planning", "tool_use", "routing", "evaluation"]

# ─────────────────────────────────────────────────────────────────────────────
# TASK: Classify each prompt. Fill in your answers below.
# Valid types: instruction, role, planning, tool_use, routing, evaluation
# ─────────────────────────────────────────────────────────────────────────────

your_classifications = {
    1:  "",   # ← Fill in type
    2:  "",
    3:  "",
    4:  "",
    5:  "",
    6:  "",
    7:  "",
    8:  "",
    9:  "",
    10: "",
}

print("Your classifications:")
for num, prompt_text in prompts_to_classify:
    answer = your_classifications.get(num, "(not filled)")
    status = "✅" if answer in VALID_TYPES else "⬜"
    print(f"  {status} [{num:02d}] {prompt_text[:65]}..." if len(prompt_text) > 65 else f"  {status} [{num:02d}] {prompt_text}")
    if answer:
        print(f"        → {answer}")

In [ ]:
# ─── LLM Auto-Classifier ─────────────────────────────────────────────────────
# Run this to see how the LLM classifies each prompt

CLASSIFIER_SYSTEM = """You are a prompt engineering expert who classifies prompts by type.
You are precise and always choose exactly one type."""

CLASSIFIER_PROMPT = """
Classify the following prompt into exactly one of these types:
- instruction: Directs the model to perform a specific task
- role: Establishes the model's identity or persona
- planning: Asks the model to decompose a goal into steps
- tool_use: Tells the model it has access to external tools
- routing: Asks the model to classify input and return a label
- evaluation: Asks the model to judge/score something against criteria

Prompt: \"{prompt}\"

Return ONLY this JSON (no other text):
{{"type": "[one of the six types]", "reason": "one sentence"}}
"""

print("🤖 LLM AUTO-CLASSIFIER RESULTS")
print("=" * 65)

llm_classifications = {}

for num, prompt_text in prompts_to_classify:
    result = call_llm(
        CLASSIFIER_PROMPT.format(prompt=prompt_text),
        system=CLASSIFIER_SYSTEM,
        max_tokens=128
    )
    try:
        clean = result.strip().replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)
        llm_type = parsed["type"]
        llm_reason = parsed["reason"]
        llm_classifications[num] = llm_type
        
        your_ans = your_classifications.get(num, "")
        match = "✅" if your_ans == llm_type else ("🔶" if your_ans else "⬜")
        print(f"\n[{num:02d}] {match} LLM says: {llm_type}")
        if your_ans:
            print(f"      You said: {your_ans}")
        print(f"      Reason: {llm_reason}")
    except:
        print(f"\n[{num:02d}] Could not parse: {result}")
    time.sleep(0.5)

In [ ]:
# ─── REVEAL: Expected classifications ────────────────────────────────────────

expected_classifications = {
    1:  "role",         # Pure identity definition. No task, no output.
    2:  "instruction",  # Specific task with action verb 'summarise'
    3:  "planning",     # Explicit decomposition request
    4:  "tool_use",     # 'calculator tool' + concrete calculation
    5:  "routing",      # Classification into fixed labels
    6:  "evaluation",   # Score + feedback against criteria
    7:  "instruction",  # Extraction task with explicit output format
    8:  "planning",     # 'What steps' = decomposition request
    9:  "routing",      # Binary/ternary label output only
    10: "evaluation",   # Judging quality of code against criteria
}

notes = {
    8: "Note: Could be 'instruction' — but 'what steps' primes the model to produce a plan, not an answer.",
    10: "Note: Could be 'instruction' — but asking to 'identify issues' against quality criteria = evaluation.",
}

print("🔍 DEBRIEF — Expected Classifications")
print("=" * 65)

correct = 0
for num, expected in expected_classifications.items():
    your = your_classifications.get(num, "")
    llm  = llm_classifications.get(num, "")
    y_match = "✅" if your == expected else ("❌" if your else "⬜")
    l_match = "✅" if llm == expected else "❌"
    if your == expected:
        correct += 1
    print(f"[{num:02d}] Expected: {expected:<12} You:{y_match}  LLM:{l_match}")
    if num in notes:
        print(f"     ℹ️  {notes[num]}")

print(f"\nYour score: {correct}/{len(expected_classifications)}")

In [ ]:
# ─── Challenge: Write your own prompt of each type ───────────────────────────
# Write one original prompt for each type, then validate with the classifier.

your_original_prompts = {
    "instruction": "",   # YOUR PROMPT HERE
    "role":        "",
    "planning":    "",
    "tool_use":    "",
    "routing":     "",
    "evaluation":  "",
}

print("🎯 CHALLENGE: Validate your original prompts")
print("=" * 55)

for intended_type, prompt_text in your_original_prompts.items():
    if not prompt_text:
        print(f"  ⬜ {intended_type}: not written yet")
        continue

    result = call_llm(
        CLASSIFIER_PROMPT.format(prompt=prompt_text),
        system=CLASSIFIER_SYSTEM,
        max_tokens=128
    )
    try:
        clean = result.strip().replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)
        detected = parsed["type"]
        match = "✅" if detected == intended_type else "❌"
        print(f"  {match} Intended: {intended_type:<12} Detected: {detected}")
        if detected != intended_type:
            print(f"     Reason: {parsed['reason']}")
            print(f"     → Revise your prompt to make the type clearer.")
    except:
        print(f"  {intended_type}: parse error")
    time.sleep(0.5)

---
# 🟣 Hour 6 Exercise — Zero-Shot vs Few-Shot + Task Decomposition

## Part A: Zero-Shot vs. Few-Shot Comparison

You will classify the same 10 customer feedback samples using two approaches, then compare the consistency of each.

**Categories:** `FEATURE_REQUEST` | `BUG_REPORT` | `GENERAL_PRAISE`

In [ ]:
# 10 customer feedback samples (some deliberately ambiguous)
feedback_samples = [
    "The app is fantastic but I wish I could export data to CSV.",
    "Every time I try to log in from my phone it crashes immediately.",
    "This tool has genuinely changed how my team works. Thank you!",
    "It would be amazing if the dashboard could show weekly trends.",
    "The reports load slowly when there are more than 1000 rows.",
    "I love the clean interface. Very intuitive design.",
    "Dark mode would make this perfect for late-night working sessions.",
    "The search doesn't return results for special characters like # or @.",
    "Been using this for 2 years — still the best tool in this space.",
    "Notifications stopped working after the last update.",
]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK: Write a ZERO-SHOT classification prompt.
# It should classify a single feedback item into one of the three categories.
# The variable {feedback} will be substituted at runtime.
# ─────────────────────────────────────────────────────────────────────────────

ZERO_SHOT_PROMPT = """
# YOUR ZERO-SHOT PROMPT HERE
# Remember: No examples. Instructions only.
# The {feedback} placeholder will be replaced with each sample.

Feedback: {feedback}

Return only the category label: FEATURE_REQUEST, BUG_REPORT, or GENERAL_PRAISE
"""

# ─────────────────────────────────────────────────────────────────────────────
# TASK: Write a FEW-SHOT classification prompt.
# It must include at least 2 examples per category (6 total).
# The variable {feedback} will be substituted at runtime.
# ─────────────────────────────────────────────────────────────────────────────

FEW_SHOT_PROMPT = """
# YOUR FEW-SHOT PROMPT HERE
# Include at least 2 examples per category.
# Format:
#   Feedback: "..."
#   Category: CATEGORY_NAME

# --- Your examples here ---

Feedback: {feedback}
Category:
"""

print("Prompts configured. Run next cell to compare them on the 10 samples.")

In [ ]:
# ─── Run both approaches and compare ─────────────────────────────────────────

VALID_LABELS = {"FEATURE_REQUEST", "BUG_REPORT", "GENERAL_PRAISE"}

def extract_label(text: str) -> str:
    """Extract the first valid label from model output."""
    text = text.strip().upper()
    for label in VALID_LABELS:
        if label in text:
            return label
    return "UNKNOWN"

zero_shot_results = []
few_shot_results  = []

print("Running classifications... (this takes ~20 seconds)")
print("=" * 70)
print(f"{'#':<3} {'Feedback (truncated)':<42} {'Zero-Shot':<16} {'Few-Shot'}")
print("-" * 70)

for i, feedback in enumerate(feedback_samples, 1):
    skip = "YOUR" in ZERO_SHOT_PROMPT or "YOUR" in FEW_SHOT_PROMPT
    if skip:
        print(f"{i:<3} {'(fill in your prompts first)':<42} {'?':<16} {'?'}")
        continue

    zs = extract_label(call_llm(ZERO_SHOT_PROMPT.format(feedback=feedback)))
    fs = extract_label(call_llm(FEW_SHOT_PROMPT.format(feedback=feedback)))

    zero_shot_results.append(zs)
    few_shot_results.append(fs)

    agree = "✅" if zs == fs else "🔶"
    trunc = feedback[:40] + "..." if len(feedback) > 40 else feedback
    print(f"{i:<3} {trunc:<42} {zs:<16} {fs}  {agree}")
    time.sleep(0.3)

if zero_shot_results:
    agreements = sum(z == f for z, f in zip(zero_shot_results, few_shot_results))
    print(f"\nAgreements: {agreements}/10")
    print(f"Disagreements: {10 - agreements}/10 — these are the ambiguous cases worth analysing.")

In [ ]:
# ─── REVEAL: Model prompts for comparison ────────────────────────────────────

MODEL_ZERO_SHOT = """
Classify the following customer feedback into exactly one of these categories:
- FEATURE_REQUEST: The user wants a new feature or capability added to the product
- BUG_REPORT: The user is reporting something that is broken or not working
- GENERAL_PRAISE: The user is expressing satisfaction without a specific request

Constraints:
- Return only the category label. No explanation.
- If the feedback contains both a complaint and praise, classify by the PRIMARY intent.

Feedback: {feedback}
"""

MODEL_FEW_SHOT = """
Classify customer feedback into: FEATURE_REQUEST, BUG_REPORT, or GENERAL_PRAISE.
Return only the label.

Feedback: "I wish I could share dashboards with external clients."
Category: FEATURE_REQUEST

Feedback: "Would love a mobile app version of this."
Category: FEATURE_REQUEST

Feedback: "The sync button doesn't work when offline."
Category: BUG_REPORT

Feedback: "Files over 50MB cause the upload to fail silently."
Category: BUG_REPORT

Feedback: "Best project management tool I've used in 10 years."
Category: GENERAL_PRAISE

Feedback: "The customer support team is incredibly responsive."
Category: GENERAL_PRAISE

Feedback: {feedback}
Category:
"""

print("🔍 DEBRIEF — Model prompts:")
print("\n── ZERO-SHOT ──")
print(MODEL_ZERO_SHOT)
print("\n── FEW-SHOT ──")
print(MODEL_FEW_SHOT)
print()
print("Key differences:")
print("  Zero-shot: Relies on the model's definition of 'bug' vs 'feature'")
print("  Few-shot:  Installs YOUR definition through examples")
print("  Notice: The ambiguous sample #5 (slow loading) may differ between them")

## Part B: Task Decomposition Pipeline

Build a multi-step pipeline that decomposes a complex goal into subtasks and then executes them in sequence.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK: Write the PLANNER prompt.
# It should take a goal and return a numbered list of concrete subtasks.
# Each subtask must be executable by a language model.
# ─────────────────────────────────────────────────────────────────────────────

PLANNER_PROMPT = """
# YOUR PLANNER PROMPT HERE
# It should:
# - Accept a {goal} variable
# - Return 4-6 numbered subtasks
# - Each subtask starts with an action verb
# - Output ONLY the numbered list

Goal: {goal}
"""

# Goal to decompose
GOAL = "Produce a competitive analysis of the top 3 project management tools for enterprise software teams."

print("Planner prompt configured.")
print(f"Goal: {GOAL}")

In [ ]:
# ─── Run the planner + execute each subtask ───────────────────────────────────

import re

def run_decomposition_pipeline(goal: str, planner_prompt: str) -> dict:
    """Run a full decomposition pipeline: plan → execute each step."""
    
    print(f"🎯 GOAL: {goal}")
    print("\n" + "=" * 65)
    print("STEP 1: PLANNING")
    print("=" * 65)
    
    if "YOUR PLANNER PROMPT" in planner_prompt:
        print("⚠️  Fill in PLANNER_PROMPT first!")
        return {}
    
    plan_output = call_llm(planner_prompt.format(goal=goal), max_tokens=512)
    print(plan_output)
    
    # Parse numbered steps
    steps = re.findall(r'^\d+\.\s+(.+)$', plan_output, re.MULTILINE)
    if not steps:
        # Fallback: split on newlines
        steps = [line.strip() for line in plan_output.split('\n')
                 if line.strip() and line.strip()[0].isdigit()]
    
    print(f"\nParsed {len(steps)} subtasks.")
    
    # Execute each step
    results = {}
    accumulated_context = f"Overall goal: {goal}\n"

    print("\n" + "=" * 65)
    print("STEP 2: EXECUTING SUBTASKS")
    print("=" * 65)
    
    for i, step in enumerate(steps, 1):
        print(f"\n▶️  Subtask {i}: {step}")
        
        executor_prompt = f"""
You are executing one step of a multi-step analysis pipeline.

Overall goal: {goal}

Previous context:
{accumulated_context}

Your task for this step:
{step}

Execute this step concisely. Stay focused on this specific task only.
Maximum 200 words.
"""
        output = call_llm(executor_prompt, max_tokens=400)
        results[i] = {"step": step, "output": output}
        accumulated_context += f"\nStep {i} ({step}):\n{output}\n"
        print(output[:300] + ("..." if len(output) > 300 else ""))
        time.sleep(0.5)
    
    return results

pipeline_results = run_decomposition_pipeline(GOAL, PLANNER_PROMPT)

In [ ]:
# ─── REVEAL: Model planner prompt ────────────────────────────────────────────

MODEL_PLANNER = """
You are a strategic research planner. Given a goal, produce a numbered list of
concrete subtasks that, executed in sequence, will achieve the goal.

Rules for each subtask:
- Must begin with an action verb (Research, Identify, Compare, Analyse, Write, etc.)
- Must produce a specific, nameable output
- Must be executable by a language model without human input
- Must build on previous steps where applicable

Return a numbered list of 4-6 subtasks. No other text.

Goal: {goal}
"""

print("🔍 DEBRIEF — Model planner prompt:")
print(MODEL_PLANNER)
print("\nKey design choices:")
print("  1. 'without human input' — makes each step truly autonomous")
print("  2. 'nameable output' — each step produces something concrete")
print("  3. 'build on previous steps' — creates coherent pipeline")
print("  4. 'No other text' — prevents preamble that breaks parsing")

---
# 🔴 Hour 7 Exercise — Iterative Refinement Loop

## The Architecture

```
output = generate(goal)
round = 0

while round < MAX_ROUNDS:
    scores, feedback = evaluate(output)
    if all scores >= THRESHOLD: break
    output = rewrite(output, feedback)
    round += 1

return output
```

In this exercise, you will implement a complete refinement loop that:
1. Generates a blog introduction
2. Evaluates it on 3 specific criteria (with scores)
3. Rewrites it based on the critique
4. Repeats until quality threshold is met or max rounds reached
5. Tracks and displays progress across rounds

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────

TOPIC = "Why software engineers should learn to build agentic AI systems"
MAX_ROUNDS = 3
QUALITY_THRESHOLD = 4   # All scores must be >= this to stop early
MAX_SCORE = 5

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TASK: Write the three prompts that power the refinement loop.
# Each has a {variable} that will be substituted at runtime.
# ─────────────────────────────────────────────────────────────────────────────

# GENERATOR: Produces the initial blog introduction
GENERATOR_PROMPT = """
# YOUR GENERATOR PROMPT HERE
# Should produce a 3-sentence blog introduction about: {topic}
# Requirements for the intro: hook the reader immediately,
# establish the problem, and hint at the solution.
# Return ONLY the introduction. No other text.

Topic: {topic}
"""

# CRITIC: Evaluates the introduction on three criteria
# IMPORTANT: Must return valid JSON with numeric scores and feedback string
CRITIC_PROMPT = """
# YOUR CRITIC PROMPT HERE
# Evaluate on three criteria, each scored 1-5:
#   1. hook_strength: Does the first sentence immediately grab attention?
#   2. clarity: Is the problem/solution clearly established?
#   3. specificity: Are concrete details used (not vague generalities)?
#
# The prompt must return ONLY valid JSON in this exact format:
# {{"hook_strength": N, "clarity": N, "specificity": N,
#   "feedback": "specific actionable improvements"}}

Introduction: {introduction}
"""

# REWRITER: Produces improved version based on critique
REWRITER_PROMPT = """
# YOUR REWRITER PROMPT HERE
# Should fix the specific issues identified in the feedback.
# Return ONLY the rewritten introduction. No other text.

Original introduction: {introduction}

Issues to fix: {feedback}
"""

print("Three prompts configured. Run next cell to start the refinement loop.")

In [ ]:
# ─── The Refinement Loop ─────────────────────────────────────────────────────

def parse_scores(critic_output: str) -> dict | None:
    """Parse the critic's JSON output. Returns None if parsing fails."""
    try:
        clean = critic_output.strip()
        # Strip markdown fences if present
        clean = clean.replace("```json", "").replace("```", "").strip()
        # Extract JSON object if surrounded by other text
        start = clean.find("{")
        end   = clean.rfind("}") + 1
        if start >= 0 and end > start:
            clean = clean[start:end]
        return json.loads(clean)
    except json.JSONDecodeError:
        return None

def score_bar(score: int, max_score: int = 5) -> str:
    """Visual score bar."""
    filled = "█" * score
    empty  = "░" * (max_score - score)
    return f"[{filled}{empty}] {score}/{max_score}"

def run_refinement_loop(
    topic: str,
    generator_prompt: str,
    critic_prompt: str,
    rewriter_prompt: str,
    max_rounds: int = 3,
    threshold: int = 4
) -> dict:
    """Execute the full refinement loop and return all rounds."""

    # Validate prompts are filled in
    for name, prompt in [("Generator", generator_prompt),
                          ("Critic",    critic_prompt),
                          ("Rewriter",  rewriter_prompt)]:
        if "YOUR" in prompt:
            print(f"⚠️  {name} prompt not filled in yet!")
            return {}

    history = []

    print("━" * 65)
    print(f"  ITERATIVE REFINEMENT LOOP")
    print(f"  Topic:     {topic}")
    print(f"  Threshold: {threshold}/{MAX_SCORE} on all criteria")
    print(f"  Max rounds: {max_rounds}")
    print("━" * 65)

    # ── Round 0: Generate initial output ─────────────────────────────────────
    print("\n🔵 GENERATING initial introduction...")
    current = call_llm(generator_prompt.format(topic=topic), max_tokens=300)

    for round_num in range(1, max_rounds + 1):
        print(f"\n{'─'*65}")
        print(f"  ROUND {round_num} of {max_rounds}")
        print(f"{'─'*65}")
        print(f"\n📝 Current text:\n{current}\n")

        # ── Evaluate ──────────────────────────────────────────────────────────
        print("🔴 EVALUATING...")
        critic_raw = call_llm(critic_prompt.format(introduction=current), max_tokens=400)
        scores = parse_scores(critic_raw)

        if scores is None:
            print(f"⚠️  Could not parse scores. Raw output:\n{critic_raw}")
            print("   → Check your CRITIC_PROMPT returns valid JSON.")
            break

        # Display scores
        criteria = ["hook_strength", "clarity", "specificity"]
        all_pass = True
        print()
        for crit in criteria:
            val = scores.get(crit, 0)
            status = "✅" if val >= threshold else "❌"
            print(f"  {status} {crit:<15} {score_bar(val)}")
            if val < threshold:
                all_pass = False

        feedback = scores.get("feedback", "")
        print(f"\n  💬 Feedback: {feedback}")

        history.append({
            "round": round_num,
            "text": current,
            "scores": {c: scores.get(c, 0) for c in criteria},
            "feedback": feedback,
            "passed": all_pass
        })

        # ── Check stopping condition ──────────────────────────────────────────
        if all_pass:
            print(f"\n✅ THRESHOLD MET — stopping after round {round_num}.")
            break

        if round_num == max_rounds:
            print(f"\n⏱️  MAX ROUNDS reached — returning best output.")
            break

        # ── Rewrite ───────────────────────────────────────────────────────────
        print("\n🟠 REWRITING...")
        current = call_llm(
            rewriter_prompt.format(introduction=current, feedback=feedback),
            max_tokens=300
        )
        time.sleep(0.5)

    # ── Progress summary ──────────────────────────────────────────────────────
    print(f"\n{'━'*65}")
    print("  PROGRESS SUMMARY")
    print(f"{'━'*65}")
    for entry in history:
        avgs = sum(entry["scores"].values()) / len(entry["scores"])
        status = "✅ PASS" if entry["passed"] else f"Avg {avgs:.1f}"
        print(f"  Round {entry['round']}: {status}  "
              f"hook={entry['scores']['hook_strength']} "
              f"clarity={entry['scores']['clarity']} "
              f"specificity={entry['scores']['specificity']}")

    print(f"\n{'━'*65}")
    print("  FINAL OUTPUT")
    print(f"{'━'*65}")
    print(current)

    return {"history": history, "final_output": current}


# ── Run the loop ──────────────────────────────────────────────────────────────
results = run_refinement_loop(
    topic=TOPIC,
    generator_prompt=GENERATOR_PROMPT,
    critic_prompt=CRITIC_PROMPT,
    rewriter_prompt=REWRITER_PROMPT,
    max_rounds=MAX_ROUNDS,
    threshold=QUALITY_THRESHOLD
)

In [ ]:
# ─── Progress chart ───────────────────────────────────────────────────────────
# Visualise score improvement across rounds (text-based chart)

if results and results.get("history"):
    print("📈 SCORE PROGRESSION")
    print("=" * 55)
    criteria = ["hook_strength", "clarity", "specificity"]

    for crit in criteria:
        print(f"\n  {crit}:")
        for entry in results["history"]:
            val = entry["scores"][crit]
            bar = "█" * val + "░" * (5 - val)
            print(f"    Round {entry['round']}: [{bar}] {val}/5")
else:
    print("Run the refinement loop first.")

In [ ]:
# ─── REVEAL: Model prompts ────────────────────────────────────────────────────

MODEL_GENERATOR = """
You are a senior technology writer who writes compelling, specific blog introductions
for software engineering audiences.

Write a 3-sentence blog introduction about the following topic.

Requirements:
- Sentence 1: Hook — challenge a common assumption or state a surprising fact
- Sentence 2: Problem — establish the specific gap or cost the reader faces
- Sentence 3: Promise — hint at what the reader will gain by continuing

Constraints:
- Do not use generic phrases like 'In today's world' or 'AI is transforming'
- Use concrete, specific language
- Maximum 80 words
- Return only the introduction

Topic: {topic}
"""

MODEL_CRITIC = """
You are a harsh editor whose job is to find weaknesses — not praise strengths.
Evaluate the following blog introduction on three criteria.

Criteria:
1. hook_strength (1-5): Does sentence 1 genuinely surprise or challenge the reader?
   1 = generic/forgettable, 5 = immediately stops the scroll
2. clarity (1-5): Is the problem and promise clearly communicated?
   1 = confusing, 5 = crystal clear in one reading
3. specificity (1-5): Are concrete details used instead of vague generalities?
   1 = pure abstraction, 5 = specific facts/numbers/scenarios

Introduction: {introduction}

Return ONLY this JSON (no markdown, no other text):
{{"hook_strength": N, "clarity": N, "specificity": N,
  "feedback": "Identify the 2 most important specific improvements, quoting the exact phrases that need changing."}}
"""

MODEL_REWRITER = """
You are a senior technology writer doing a targeted revision of a blog introduction.

Rewrite the introduction below, fixing ONLY the specific issues in the feedback.
Do not change what is already working. Be surgical.

Original introduction:
{introduction}

Issues to fix:
{feedback}

Constraints:
- Keep the 3-sentence structure
- Maximum 80 words
- Return only the rewritten introduction
"""

print("🔍 DEBRIEF — Model prompts for the refinement loop:")
print("\nKey design choices:")
print("  Generator: Structured 3-sentence requirement prevents vague intros")
print("  Critic: Adversarial persona ('find weaknesses') reduces sycophancy")
print("  Critic: 'quoting exact phrases' forces specific, actionable feedback")
print("  Rewriter: 'Be surgical' preserves what's working")
print("\nTry running the loop with the model prompts to compare quality:")
print()

# Uncomment to run with model prompts:
# model_results = run_refinement_loop(
#     topic=TOPIC,
#     generator_prompt=MODEL_GENERATOR,
#     critic_prompt=MODEL_CRITIC,
#     rewriter_prompt=MODEL_REWRITER,
# )

---
# 🏆 Bonus Challenge — Combine All Four Techniques

Build a **mini research assistant** that:
1. Uses a **planning prompt** to decompose a research question into subtasks
2. Uses a **few-shot classification** to identify the type of each subtask
3. Executes each subtask with a **structured instruction prompt**
4. Runs the final output through a **refinement loop** until quality threshold is met

This is a preview of the agentic patterns you will implement in Modules 4 and 5.

In [ ]:
# ─── Mini Research Assistant ──────────────────────────────────────────────────
# This combines all four techniques from Module 2.
# Study this code and fill in the missing prompts.

RESEARCH_QUESTION = "What are the key architectural patterns in agentic AI systems?"

# ── Planner ────────────────────────────────────────────────────────────────────
MINI_PLANNER = """
You are a research planner. Decompose this research question into 3 concrete
research subtasks. Each must be answerable by a language model from its training.
Return a numbered list only.

Research question: {question}
"""

# ── Executor ───────────────────────────────────────────────────────────────────
MINI_EXECUTOR = """
You are a technical research assistant. Answer the following research subtask
clearly and specifically. Use concrete examples. Maximum 150 words.

Subtask: {subtask}
Context from previous steps: {context}
"""

# ── Synthesiser (with refinement) ─────────────────────────────────────────────
MINI_SYNTH = """
You are a technical writer. Synthesise the following research findings into a
coherent 3-paragraph summary. Use clear structure and concrete examples.

Research findings:
{findings}
"""

MINI_CRITIC = """
Evaluate this research summary on:
1. accuracy (1-5): Are claims technically sound?
2. structure (1-5): Is it logically organised?
3. concreteness (1-5): Are abstract claims supported by examples?

Summary: {text}

Return ONLY JSON:
{{"accuracy": N, "structure": N, "concreteness": N, "feedback": "..."}}
"""

MINI_REWRITER = """
Improve this research summary based on this feedback: {feedback}

Original summary:
{text}

Return only the improved summary.
"""

print("🚀 Running mini research assistant...")
print(f"Question: {RESEARCH_QUESTION}")
print("=" * 65)

# Step 1: Plan
print("\n📋 Planning...")
plan = call_llm(MINI_PLANNER.format(question=RESEARCH_QUESTION), max_tokens=256)
print(plan)

# Step 2: Parse and execute
subtasks = [line.strip() for line in plan.split("\n") 
            if line.strip() and line.strip()[0].isdigit()]

findings = []
ctx = ""

for subtask in subtasks:
    print(f"\n▶️  Executing: {subtask}")
    result = call_llm(MINI_EXECUTOR.format(subtask=subtask, context=ctx), max_tokens=300)
    findings.append(f"{subtask}:\n{result}")
    ctx += f"\n{result[:200]}"
    print(result[:200] + "..." if len(result) > 200 else result)
    time.sleep(0.5)

# Step 3: Synthesise
print("\n" + "=" * 65)
print("📝 Synthesising + refining...")
print("=" * 65)

summary = call_llm(MINI_SYNTH.format(findings="\n".join(findings)), max_tokens=600)

# Step 4: Refine (1 round for the bonus)
for r in range(2):
    scores_raw = call_llm(MINI_CRITIC.format(text=summary), max_tokens=256)
    scores = parse_scores(scores_raw)
    if scores:
        a, s, c = scores.get('accuracy',0), scores.get('structure',0), scores.get('concreteness',0)
        print(f"\nRound {r+1} scores: accuracy={a} structure={s} concreteness={c}")
        if min(a, s, c) >= 4:
            break
        feedback = scores.get('feedback', '')
        print(f"Feedback: {feedback}")
        summary = call_llm(MINI_REWRITER.format(feedback=feedback, text=summary), max_tokens=600)
    time.sleep(0.5)

print("\n" + "=" * 65)
print("FINAL SUMMARY")
print("=" * 65)
print(summary)

---
## 🎓 Module 2 Complete — What You've Built

| Exercise | Skill Demonstrated | Agentic Pattern Preview |
|----------|-------------------|-------------------------|
| Hour 4: Prompt Anatomy | Structured prompt engineering | Foundation of every agent |
| Hour 5: Prompt Types | Classifying by function | Maps to agent roles |
| Hour 6: Techniques | Zero-shot → Few-shot → Decomposition | Planning pattern |
| Hour 7: Refinement Loop | Generate → Evaluate → Rewrite | Reflection pattern |
| Bonus | Combined pipeline | Mini agentic workflow |

### What's next: Module 3 — Agent Types and Components
You'll move from prompts to agents — implementing Router, Planner, Executor, and Critic agents in Python.

---
*Module 2 Exercises — Agentic AI Professional Course*  
*Complete all exercises before attending the Module 3 live session.*